In [1]:
!pip install pandas

You should consider upgrading via the '/usr/bin/python3 -m pip install --upgrade pip' command.


In [2]:
import pandas as pd

In [3]:
# Archivos a leer, que contienen todos los grafos y features de los nodos
# Cada nodo está identificado con un prefijo que indica la captura de la cual proviene

edges_file_orig = "/mnt/training_GRAFOS.pkts.ncol"
features_file_orig = "/mnt/training_FEATURES.pkts.SINnorm.csv"

# Se crearán los siguientes archivos, luego de cambiar los nodos por un número entero
edges_file_int = "/mnt/training_GRAFOS_INT.pkts.ncol"
features_file_int = "/mnt/training_FEATURES_INT.pkts.SINnorm.csv"

In [6]:
# Leemos archivo con las features de cada nodo
# Cambiamos "background" por "normal" en la columna "label"
usecols = ["node","ID","OD","IDW","ODW","label"]
features_orig = pd.read_csv(features_file_orig, sep=",", header=0, usecols=lambda c: c in set(usecols))
features_orig.loc[:,"label"] = features_orig.loc[:,"label"].replace(to_replace="background", value="normal").copy()

# Generamos diccionario para las clases y nodos
# A cada nodo (y clase) le corresponde un número entero
class_idx = {name: idx for idx, name in enumerate(sorted(features_orig["label"].unique()))}
node_idx = {name: idx for idx, name in enumerate(sorted(features_orig["node"].unique()))}

In [23]:
# Cambiamos los nodos y clases por su correspondiente número entero, en las features y en los grafos
features_int = features_orig.copy()
features_int["node"] = features_int["node"].apply(lambda name: node_idx[name])
features_int["label"] = features_int["label"].apply(lambda value: class_idx[value])

grafos_orig = pd.read_csv(edges_file_orig, sep=" ", header=None, names=["source", "target", "weight"],)
grafos_int = grafos_orig.copy()
grafos_int["source"] = grafos_int["source"].apply(lambda name: node_idx[name])
grafos_int["target"] = grafos_int["target"].apply(lambda name: node_idx[name])

# Generamos nuevo archivo con los identificadores enteros
grafos.to_csv(edges_file_int, sep=" ", header=False, index=False)
features.to_csv(features_file_int, index=False)

In [22]:
grafos_orig.iloc[1234211:(1234211+877634),]

,source,target,weight
1234211,2-0.0.0.0,2-255.255.255.255,1518
1234212,2-1.1.1.45,2-147.32.84.229,7
1234213,2-1.11.62.138,2-147.32.84.229,1
1234214,2-1.112.0.241,2-147.32.84.229,1
1234215,2-1.112.104.74,2-147.32.84.229,1
...,...,...,...
2111840,2-99.99.85.224,2-147.32.84.229,44
2111841,2-99.99.86.247,2-147.32.84.229,31
2111842,2-99.99.86.247,2-147.32.86.165,56
2111843,2-fe80::d036:46ef:ad35:fc09,2-ff02::1:3,2


In [20]:
grafos_int.iloc[1234211:(1234211+877634),]

,source,target,weight
1234211,1157271,1343205,1518
1234212,1157272,1232256,7
1234213,1157273,1232256,1
1234214,1157274,1232256,1
1234215,1157275,1232256,1
...,...,...,...
2111840,1597840,1232256,44
2111841,1597841,1232256,31
2111842,1597841,1232697,56
2111843,1597842,1597844,2


In [24]:
capturas = ["20110810","20110811","20110812","20110815","20110815-2","20110816","20110816-2","20110818","20110818-2","20110815-3"]

edges_all = pd.read_csv(edges_file_int, sep=" ", header=None)
features_all = pd.read_csv(features_file_int, sep=",", header=0)

lineasEdges = [1234211,877634,931570,391876,85782,228580,78940,439620,86351,666207]
# for i in `seq 1 13`; do echo $i; awk -F "," '{ print $1 }' training_GRAFOS.pkts.ncol | grep -e ^"$i-" | wc -l; done

lineasFeatures = [605195,440574,430265,184901,41399,106580,37943,196686,41712,313678]
# for i in `seq 1 13`; do echo $i; awk -F "," '{ print $1 }' training_FEATURES.pkts.SINnorm.csv | grep -e ^"$i-" | wc -l; done


inicioLE = 0
inicioLF = 0
for i in range(len(capturas)):
    tmpE = edges_all.iloc[inicioLE:(inicioLE+lineasEdges[i]),:].copy()
    tmpE.to_csv("/mnt/edges_int/edges_INT_capture"+str(capturas[i])+".ncol", sep=" ", header=False, index=False)
    tmpF = features_all.iloc[inicioLF:(inicioLF+lineasFeatures[i]),:].copy()
    tmpF.to_csv("/mnt/features_int/features_INT_capture"+str(capturas[i])+".csv", index=False)
    inicioLE += lineasEdges[i]
    inicioLF += lineasFeatures[i]
    